In [1]:
import sys
!{sys.executable} -m pip install -U scikit-learn nltk spacy -q
!{sys.executable} -m spacy download es_core_news_sm -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
label-studio-sdk 2.0.16 requires opencv-python-headless<5.0.0,>=4.12.0, which is not installed.
streamlit 1.45.1 requires pillow<12,>=7.1.0, but you have pillow 12.1.0 which is incompatible.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
label-studio 1.22.0 requires numpy<3.0.0,>=2.2.6, but you have numpy 2.1.3 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [ ]:
import pandas as pd

df = pd.read_csv('df_total.csv', encoding='UTF-8')

print('Shape:', df.shape)
print('Columnas:', df.columns.tolist())
df.head()

Shape: (1217, 3)
Columnas: ['url', 'news', 'Type']


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra


In [4]:
print('Distribución de categorías:')
print(df['Type'].value_counts())

print(f"\nNoticia original [3]:\n{df['news'][3]}")

Distribución de categorías:
Type
Macroeconomia     340
Alianzas          247
Innovacion        195
Regulaciones      142
Sostenibilidad    137
Otra              130
Reputacion         26
Name: count, dtype: int64

Noticia original [3]:
Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual respecto al avance del 30 de marzo y se sitúa 22 puntos por encima del dato de febrero que ascendió al 76.De esos 22 puntos de diferencia la mayor parte la colocó el grupo de la vivienda 09 puntos por la subida de la electricidad y el del transporte 07 puntos por el alza de los carburantes. También impulsaron el IPC de marzo el aumento de los precios de la restauración y los servicios de alojamiento y al encarecimiento generalizado de los alimentos especialmente del pescado y el marisco de la carne de las legumbres y hortalizas y de la leche el queso y los huevos.Sin tener en cuenta la rebaja del impuesto espe

In [6]:
from sklearn.model_selection import train_test_split

X = df['news']  # texto
y = df['Type']  # categoría

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Entrenamiento: {len(X_train)} noticias')
print(f'Prueba:        {len(X_test)} noticias')

Entrenamiento: 973 noticias
Prueba:        244 noticias


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed  = vectorizer.transform(X_test)

print('Forma de la matriz:', X_train_transformed.shape)

X_train_transformed_dense = X_train_transformed.toarray()
print('\nPrimeras 10 palabras del vocabulario:')
print(vectorizer.get_feature_names_out()[:10])
print('\nConteos (5 noticias x 10 palabras):')
print(X_train_transformed_dense[:5, :10])

Forma de la matriz: (973, 26512)

Primeras 10 palabras del vocabulario:
['00' '000' '000a' '000m' '000mentre' '001' '002' '003' '004' '005']

Conteos (5 noticias x 10 palabras):
[[0 2 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

model = MultinomialNB()
model.fit(X_train_transformed, y_train)

y_pred = model.predict(X_test_transformed)

acc_base = metrics.accuracy_score(y_test, y_pred)
print(f'Precisión BASE: {acc_base:.4f} ({acc_base*100:.2f}%)')
print(metrics.classification_report(y_test, y_pred))

Precisión BASE: 0.7992 (79.92%)
                precision    recall  f1-score   support

      Alianzas       0.85      0.79      0.82        52
    Innovacion       0.63      1.00      0.78        33
 Macroeconomia       0.81      0.93      0.87        73
          Otra       0.83      0.52      0.64        29
  Regulaciones       1.00      0.62      0.77        24
    Reputacion       0.00      0.00      0.00         9
Sostenibilidad       0.85      0.96      0.90        24

      accuracy                           0.80       244
     macro avg       0.71      0.69      0.68       244
  weighted avg       0.79      0.80      0.78       244



/home/ciabd01/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ciabd01/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ciabd01/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]

In [10]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

stemmer = SnowballStemmer('spanish')

def tokenize_and_stem(text):
    tokens = word_tokenize(text.lower())
    stems = [stemmer.stem(token) for token in tokens if token.isalpha()]
    return ' '.join(stems)

# Ver diferencia
print('ORIGINAL:  ', df['news'][3][:200])
print('\nSTEMMING:  ', tokenize_and_stem(df['news'][3])[:200])

ORIGINAL:   Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual respecto al avance del 30 de marzo y se sitúa 22 punt

STEMMING:   con el dat de marz el ipc interanual encaden su decimoquint tas posit consecut la inflacion public por el ine se ha manten igual respect al avanc del de marz y se situ punt por encim del dat de febrer


In [11]:
print('Aplicando stemming...')
df['news_stemmer'] = df['news'].apply(tokenize_and_stem)

X = df['news_stemmer']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

acc_stem = metrics.accuracy_score(y_test, y_pred)
print(f'✅ Precisión STEMMING: {acc_stem:.4f} ({acc_stem*100:.2f}%)')
print(f'   Diferencia vs base: {(acc_stem - acc_base)*100:+.2f}%')

Aplicando stemming...
✅ Precisión STEMMING: 0.8279 (82.79%)
   Diferencia vs base: +2.87%


In [12]:
import spacy
nlp = spacy.load('es_core_news_sm')

def lemmatize_text(text):
    doc = nlp(text.lower())
    lemmas = [token.lemma_ for token in doc if token.is_alpha]
    return ' '.join(lemmas)

# Ver diferencia
print('ORIGINAL:      ', df['news'][3][:200])
print('\nLEMATIZACIÓN:  ', lemmatize_text(df['news'][3])[:200])

ORIGINAL:       Con el dato de marzo el IPC interanual encadena su decimoquinta tasa positiva consecutiva. La inflación publicada por el INE se ha mantenido igual respecto al avance del 30 de marzo y se sitúa 22 punt

LEMATIZACIÓN:   con el dato de marzo el ipc interanual encadenar su decimoquinto tasa positivo consecutivo el inflación publicado por el ine él haber mantener igual respecto al avance del de marzo y él situar punto p


In [15]:
print('Aplicando lematización... (puede tardar unos minutos)')
df['news_lemma'] = df['news'].apply(lemmatize_text)

X = df['news_lemma']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

acc_lemma = metrics.accuracy_score(y_test, y_pred)
print(f'Precisión LEMATIZACIÓN: {acc_lemma:.4f} ({acc_lemma*100:.2f}%)')
print(f'   Diferencia vs base:     {(acc_lemma - acc_base)*100:+.2f}%')
print(f'   Diferencia vs stemming: {(acc_lemma - acc_stem)*100:+.2f}%')

Aplicando lematización... (puede tardar unos minutos)
Precisión LEMATIZACIÓN: 0.8320 (83.20%)
   Diferencia vs base:     +3.28%
   Diferencia vs stemming: +0.41%


In [14]:
import pandas as pd

resumen = pd.DataFrame({
    'Método':    ['Base', 'Stemming', 'Lematización'],
    'Precisión': [acc_base, acc_stem, acc_lemma]
})
resumen['Precisión %'] = resumen['Precisión'].apply(lambda x: f'{x*100:.2f}%')
resumen['vs Base']     = resumen['Precisión'].apply(lambda x: f'{(x-acc_base)*100:+.2f}%')

print(resumen.to_string(index=False))
print('\n→ Mejor método:', resumen.loc[resumen['Precisión'].idxmax(), 'Método'])

      Método  Precisión Precisión % vs Base
        Base   0.799180      79.92%  +0.00%
    Stemming   0.827869      82.79%  +2.87%
Lematización   0.831967      83.20%  +3.28%

→ Mejor método: Lematización
